Number of NHDAs that have construction start time in the specific calendar year.

In [ ]:
import math
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Polygon, Patch
from pathlib import Path

# -----------------------------------------------------------------------------
# Input / Output
# -----------------------------------------------------------------------------
gpkg_file = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"

OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\Maps_Baujahr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CRS = "EPSG:25832"
GRID_STEP_M = 50000
MAP_PADDING_M = 10000

COLOR_THEME = ['#fff0f3', '#ffccd5', '#ffb3c1', '#ff8fa3', '#ff758f', '#ff758f', '#ff4d6d', "#c9184a", "#a4133c", "#800f2f", "#590d22"]
CUSTOM_CMAP = LinearSegmentedColormap.from_list('custom_orange_brown', COLOR_THEME)

YEAR_COL = "construction_start_year"

# -----------------------------------------------------------------------------
# Map styling helpers (identisch zum Morphologie-Skript)
# -----------------------------------------------------------------------------

def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{int(round(y / 1000))}"))
    ax.tick_params(axis="both", which="major", labelsize=10, length=0, colors="#9B9999")

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker="+", s=24, linewidths=1.0, color="#8a8a8a", alpha=0.85, zorder=4, clip_on=True)


def add_north_arrow(ax):
    x, y = 0.94, 0.88
    width, height = 0.025, 0.09

    left_triangle = Polygon(
        [(x, y + height), (x - width, y), (x, y + height * 0.30)],
        closed=True, transform=ax.transAxes,
        facecolor="#222222", edgecolor="#222222", linewidth=1.0, zorder=10,
    )
    right_triangle = Polygon(
        [(x, y + height), (x + width, y), (x, y + height * 0.30)],
        closed=True, transform=ax.transAxes,
        facecolor="white", edgecolor="#222222", linewidth=1.0, zorder=10,
    )
    ax.add_patch(left_triangle)
    ax.add_patch(right_triangle)
    ax.text(x, y + height + 0.018, "N", transform=ax.transAxes, ha="center", va="bottom",
            fontsize=14, fontweight="bold", color="#222222", zorder=10)


def add_scale_bar(ax):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0
    total_km, segment_km = 50, 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000
    x_start = x1 - span_x * 0.35
    y_start = y0 + span_y * 0.060
    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color="#222222", linewidth=1.3, zorder=8)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color="#222222", linewidth=1.0, zorder=8)
    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, "0", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + segment_len, txt_y, "25", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + bar_len, txt_y, "50 km", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlabel("Easting (km) - UTM 32N", fontsize=11, color="#555555")
    ax.set_ylabel("Northing (km) - UTM 32N", fontsize=11, color="#555555")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)


# -----------------------------------------------------------------------------
# Daten laden
# -----------------------------------------------------------------------------
gdf = gpd.read_file(gpkg_file)

if YEAR_COL not in gdf.columns:
    raise KeyError(f"Spalte '{YEAR_COL}' nicht in {gpkg_file} gefunden.")

gdf = gdf[gdf.geometry.notna()].copy()
gdf = gdf.to_crs(TARGET_CRS)

# Baujahr-Werte als String normalisieren (Jahreszahl -> "2015", AUC bleibt "AUC_2015")
def normalize_year_value(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.upper().startswith("AUC"):
        return s.upper().replace(" ", "_")
    try:
        return str(int(float(s)))
    except (ValueError, TypeError):
        return s

gdf["year_group"] = gdf[YEAR_COL].apply(normalize_year_value)
gdf = gdf[gdf["year_group"].notna()].copy()

# Landkreise laden (nur Bayern, wie im Morphologie-Skript)
gdf_lk = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE).to_crs(TARGET_CRS)
# Keep only Bavarian districts (ARS starts with 09)
gdf_lk = gdf_lk[
    gdf_lk["Regionalschlüssel_ARS"].astype(str).str.startswith("09")
].copy()

LK_KEY_CANDIDATES = ["ARS", "ags", "AGS", "AGS_0", "krs_code", "kreis_code", "SCHLUESSEL", "KRS", "Regionalschlüssel_ARS"]

def _find_key_col(gdf_, candidates):
    cols_lower = {c.lower(): c for c in gdf_.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

lk_key_col = _find_key_col(gdf_lk, LK_KEY_CANDIDATES)
if lk_key_col is None:
    raise KeyError(f"Keine Landkreis-ID-Spalte gefunden unter {LK_KEY_CANDIDATES}.")
gdf_lk["lk_code"] = gdf_lk[lk_key_col].astype(str).str.strip()

# -----------------------------------------------------------------------------
# Spatial join: NHDA-Zentroid -> Landkreis
# -----------------------------------------------------------------------------
centroids = gdf[["geometry"]].copy()
centroids["geometry"] = centroids.geometry.centroid

joined = gpd.sjoin(centroids, gdf_lk[["lk_code", "geometry"]], how="left", predicate="within")
joined = joined[~joined.index.duplicated(keep="first")]
gdf["lk_code"] = joined["lk_code"]

n_unmatched = gdf["lk_code"].isna().sum()
if n_unmatched:
    print(f"Warnung: {n_unmatched} NHDA-Polygone konnten keinem Landkreis zugeordnet werden.")

# -----------------------------------------------------------------------------
# Sortierte Liste der Jahresgruppen: erst Jahreszahlen aufsteigend, dann AUC_*
# -----------------------------------------------------------------------------
def sort_key(v):
    if v.upper().startswith("AUC"):
        return (1, v)
    try:
        return (0, int(v))
    except ValueError:
        return (2, v)

year_groups = sorted(gdf["year_group"].unique(), key=sort_key)


def label_for_group(g):
    if g.upper().startswith("AUC"):
        return g.replace("AUC_", "AUC ")
    return f"Construction Start: {g}"


def plot_year_count_map(year_group):
    subset = gdf[gdf["year_group"] == year_group]
    counts = subset.groupby("lk_code").size().rename("count").reset_index()
    map_df = gdf_lk.merge(counts, on="lk_code", how="left")

    values = map_df["count"].dropna()
    if values.empty:
        print(f"Warnung: Keine NHDA für {year_group} - Karte wird übersprungen.")
        return

    vmin = 0
    vmax = values.max()
    if vmax == 0:
        vmax = 1

    fig, ax = plt.subplots(figsize=(8.8, 9.2))
    add_scientific_frame(ax, gdf_lk)
    map_df.plot(
        ax=ax,
        column="count",
        cmap=CUSTOM_CMAP,
        linewidth=0.35,
        edgecolor="#5a5a5a",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={"color": "#d9d9d9", "edgecolor": "#b7b7b7", "label": "No data"},
    )

    label = label_for_group(year_group)
    ax.set_title(f"{label}", fontsize=18, pad=14)
    add_north_arrow(ax)
    add_scale_bar(ax)

    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=CUSTOM_CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label("Number of NHDA", fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    missing_handle = Patch(facecolor="#d9d9d9", edgecolor="#b7b7b7", label="No data")
    ax.legend(handles=[missing_handle], loc="upper left", frameon=True, framealpha=0.95,
              facecolor="white", edgecolor="#cccccc", fontsize=10)

    safe_name = year_group.replace(" ", "_")
    out_file = OUTPUT_DIR / f"nhda_count_{safe_name}_landkreis.jpg"
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_file}")


for yg in year_groups:
    plot_year_count_map(yg)